# Retail dbt Test Generation Pipeline

End-to-end pipeline: PostgreSQL retail data → direct schema extraction from PostgreSQL → Ollama LLM → dbt test generation → dbt test run → results preview.

In [ ]:
# Needed to clear the .env variabel path

import os

from dotenv import load_dotenv

os.environ.pop("POST_TEST", None)
os.environ.pop("POSTGRES_HOST", None)

load_dotenv("../.env", override=True)


In [ ]:
# Cell 1: Setup & Connectivity Check
import os
import psycopg2
import requests

from dotenv import load_dotenv

load_dotenv(os.path.join(os.getcwd(), "..", ".env"))

# Config from env
PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = int(os.getenv("POSTGRES_PORT", "5435"))
PG_USER = os.getenv("POSTGRES_USER", "retail")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "retail123")
PG_DB = os.getenv("POSTGRES_DB", "retail")
PG_SCHEMA = os.getenv("POSTGRES_SCHEMA", "public")

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "localhost")
OLLAMA_PORT = int(os.getenv("OLLAMA_PORT", "11434"))
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5-coder:3b")

DBT_PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "dbt_project"))

# Connectivity checks
checks = {}

# PostgreSQL
try:
    conn = psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        user=PG_USER,
        password=PG_PASSWORD,
        dbname=PG_DB,
        connect_timeout=5,
    )
    conn.close()
    checks["PostgreSQL"] = True
except Exception as e:
    checks["PostgreSQL"] = str(e)

# Ollama
try:
    r = requests.get(f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/tags", timeout=5)
    checks["Ollama"] = r.status_code == 200
except Exception as e:
    checks["Ollama"] = str(e)

print("=== Connectivity Checks ===")
for service, status in checks.items():
    icon = "OK" if status is True else "FAIL"
    print(f"  [{icon}] {service}: {status if status is not True else 'connected'}")


In [ ]:
# Cell 2: Load Table Metadata from PostgreSQL
import sys
from pathlib import Path

package_root = Path.cwd().parent
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

from pipeline.dbt_generator import list_tables_with_columns


def load_table_metadata():
    conn = psycopg2.connect(
        host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
    )
    try:
        return list_tables_with_columns(conn, schema=PG_SCHEMA)
    finally:
        conn.close()


table_metadata = load_table_metadata()
print(
    f"Loaded metadata for {len(table_metadata)} tables from PostgreSQL schema '{PG_SCHEMA}'."
)


In [ ]:
# Cell 3: Extract Schema from PostgreSQL
import pandas as pd

if table_metadata:
    schema_df = pd.DataFrame(
        [
            {
                "table_name": t["name"],
                "column_count": len(t["columns"]),
                "fqn": t["fqn"],
            }
            for t in table_metadata
        ]
    )
    print(f"Tables extracted from PostgreSQL:")
    display(schema_df)

    # Build lookup dict: table_name -> columns
    table_columns = {t["name"]: t["columns"] for t in table_metadata}
    table_names = [t["name"] for t in table_metadata]
else:
    print(
        "No tables found. Make sure seed.py ran and the PostgreSQL schema is populated."
    )
    table_columns = {}
    table_names = []


In [ ]:
# Cell 4: Sample Data Preview
from pipeline.dbt_generator import get_sample_rows

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
)

print("Sample data (5 rows per table):")
for table in table_names[:5]:  # preview first 5 tables
    rows = get_sample_rows(conn, table, limit=5)
    if rows:
        print(f"\n--- {table} ---")
        display(pd.DataFrame(rows))


In [ ]:
# Cell 5: Compute Column Statistics
from pipeline.dbt_generator import compute_column_stats

print("Computing column statistics...")
all_stats = {}  # table_name -> col_name -> stats

for table in table_names:
    cols = table_columns.get(table, [])
    if cols:
        stats = compute_column_stats(conn, table, cols)
        all_stats[table] = stats
        print(f"  {table}: {len(cols)} columns analyzed")

# Build summary DataFrame
rows = []
for table, col_stats in all_stats.items():
    for col_name, s in col_stats.items():
        rows.append(
            {
                "table": table,
                "column": col_name,
                "null_rate": round(s.get("null_rate", 0), 4),
                "uniqueness_score": round(s.get("uniqueness_score", 0), 4),
                "distinct_count": s.get("distinct_count", 0),
                "zero_count": s.get("zero_count"),
                "min_val": s.get("min_val"),
                "max_val": s.get("max_val"),
                "total_count": s.get("total_count", 0),
            }
        )

stats_df = pd.DataFrame(rows).sort_values("null_rate", ascending=False)
print(f"\nColumn stats summary (sorted by null_rate):")
display(stats_df.head(30))


In [ ]:
# Cell 6: LLM Test Generation via Ollama
from pipeline.llm_client import OllamaClient, build_prompt, parse_yaml_output

ollama = OllamaClient(OLLAMA_HOST, OLLAMA_PORT, OLLAMA_MODEL)
if not ollama.health():
    print("ERROR: Ollama not reachable at", f"http://{OLLAMA_HOST}:{OLLAMA_PORT}")
else:
    print(f"Ollama ready. Model: {OLLAMA_MODEL}")

per_table_schemas = []

for table in table_names[:2]:
    cols = table_columns.get(table, [])
    stats = all_stats.get(table, {})
    sample_rows = get_sample_rows(conn, table, limit=10)

    prompt = build_prompt(table, cols, stats, sample_rows)

    print(f"Generating tests for {table}...")
    raw = ollama.generate(prompt)

    parsed = parse_yaml_output(raw)
    if parsed:
        models = parsed.get("models", [])
        test_count = sum(len(m.get("columns", [])) for m in models)
        per_table_schemas.append(parsed)
        print(
            f"  Generated schema for {table}: {len(models)} models, ~{test_count} column entries"
        )
    else:
        print(f"  WARNING: Could not parse YAML for {table}. Raw output:")
        print(raw[:200])

print(f"\nGenerated schemas for {len(per_table_schemas)}/{len(table_names)} tables.")


In [ ]:
# Cell 7: Write Generated Files to dbt Project
from pipeline.dbt_generator import write_dbt_files, merge_schema_ymls

merged_yml = merge_schema_ymls(per_table_schemas)

print(f"Writing dbt files to: {DBT_PROJECT_DIR}")
write_dbt_files(
    dbt_project_dir=DBT_PROJECT_DIR,
    tables=table_names,
    merged_schema_yml=merged_yml,
    source_name="retail",
)
print("Done writing dbt files.")


In [ ]:
# Cell 8: Run dbt (deps → run → test → docs generate)
import subprocess


def run_dbt_cmd(args, cwd):
    cmd = ["dbt"] + args + ["--profiles-dir", "."]
    print(f"Running: dbt {' '.join(args)}")
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-3000:])  # last 3000 chars
    if result.stderr:
        print("STDERR:", result.stderr[-1000:])
    return result.returncode


print("=== dbt deps ===")
run_dbt_cmd(["deps"], DBT_PROJECT_DIR)

print("\n=== dbt run (staging) ===")
run_dbt_cmd(["run", "--select", "staging"], DBT_PROJECT_DIR)

print("\n=== dbt test (with store-failures) ===")
run_dbt_cmd(["test", "--store-failures", "--select", "staging"], DBT_PROJECT_DIR)

print("\n=== dbt docs generate ===")
run_dbt_cmd(["docs", "generate"], DBT_PROJECT_DIR)

print("\ndbt commands complete.")


In [ ]:
# Cell 9: Results Dashboard
import json

run_results_path = os.path.join(DBT_PROJECT_DIR, "target", "run_results.json")

if not os.path.exists(run_results_path):
    print("No run_results.json found. Run Cell 8 first.")
else:
    with open(run_results_path) as f:
        run_results = json.load(f)

    results = run_results.get("results", [])
    rows = []
    for r in results:
        node = r.get("unique_id", "")
        node_name = node.split(".")[-1]
        rows.append(
            {
                "test_name": node_name,
                "status": r.get("status", "unknown"),
                "execution_time_s": round(r.get("execution_time", 0), 2),
                "failures": r.get("failures", 0),
                "message": (r.get("message") or "")[:80],
            }
        )

    results_df = pd.DataFrame(rows).sort_values(
        ["status", "failures"], ascending=[True, False]
    )

    total = len(results_df)
    passed = (results_df["status"] == "pass").sum()
    failed = (results_df["status"] == "fail").sum()

    print(f"=== dbt Test Results ===")
    print(f"Total: {total} | Passed: {passed} | Failed: {failed}")
    print()
    display(results_df)


In [ ]:
# Cell 10: Column Metrics Chart
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.figsize"] = (12, 5)

# Chart 1: Test results by status
if "results_df" in dir() and not results_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    status_counts = results_df["status"].value_counts()
    colors = {
        "pass": "#4CAF50",
        "fail": "#F44336",
        "warn": "#FF9800",
        "error": "#9C27B0",
    }
    bar_colors = [colors.get(s, "#999") for s in status_counts.index]
    ax1.bar(status_counts.index, status_counts.values, color=bar_colors)
    ax1.set_title("dbt Tests by Status")
    ax1.set_xlabel("Status")
    ax1.set_ylabel("Count")
    for i, (idx, val) in enumerate(status_counts.items()):
        ax1.text(i, val + 0.1, str(val), ha="center", va="bottom", fontweight="bold")

    # Chart 2: Top 10 columns by null_rate
    if not stats_df.empty:
        top_nulls = stats_df[stats_df["null_rate"] > 0].nlargest(10, "null_rate")
        if not top_nulls.empty:
            labels = top_nulls["table"] + "." + top_nulls["column"]
            ax2.barh(labels, top_nulls["null_rate"] * 100, color="#F44336", alpha=0.7)
            ax2.set_title("Top 10 Columns by Null Rate")
            ax2.set_xlabel("Null Rate (%)")
            ax2.invert_yaxis()
        else:
            ax2.text(
                0.5,
                0.5,
                "No nulls found!",
                ha="center",
                va="center",
                transform=ax2.transAxes,
            )
            ax2.set_title("Null Rates")

    plt.tight_layout()
    plt.savefig(
        os.path.join("..", "dbt_test_results.png"), dpi=100, bbox_inches="tight"
    )
    plt.show()
    print("Chart saved to dbt_test_results.png")


In [ ]:
# Cell 11: Links & Summary
import subprocess

# Start dbt docs serve in background
docs_proc = subprocess.Popen(
    ["dbt", "docs", "serve", "--port", "8082", "--profiles-dir", "."],
    cwd=DBT_PROJECT_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("=== Pipeline Complete ===")
print(f"Tables discovered from PostgreSQL: {len(table_names)}")
print(f"dbt schemas generated by LLM: {len(per_table_schemas)}")

if "results_df" in dir() and not results_df.empty:
    passed = (results_df["status"] == "pass").sum()
    failed = (results_df["status"] == "fail").sum()
    total = len(results_df)
    print(f"dbt tests: {total} total | {passed} passed | {failed} failed")

print()
print("Links:")
print(f"  dbt Docs:       http://localhost:8082")
print(f"  PostgreSQL:     localhost:{PG_PORT}")
print(f"  Ollama:         http://localhost:{OLLAMA_PORT}")
print()
print("To stop dbt docs server: docs_proc.terminate()")

# Close DB connection
if conn and not conn.closed:
    conn.close()
